# 02 — Certified random MPUs, Mode I (row resolution)

Mode I builds $A$ on the edges of a digraph over the $\chi$ bond values. Every row $v$ satisfies the **row-resolution identity**

$$\sum_u A[v,u]^\dagger A[v,u]=\mathbb 1_d,\qquad A[v,u]^\dagger A[v,u']=0\quad(u\neq u'),$$

and its mirror. With the single-vertex readout $\ell=e_{0}$, $r=(1,\dots,1)$, the chain is unitary for **every** $N$ (thesis `thm:mode1-unitarity`; paper `thm:main1`). The certificate is local and needs no circuit-level or finite-$N$ check.

The sampler draws Haar unitaries $V,W$ per row and sets $A[v,u_i]=V P_i W^\dagger$, where the $P_i$ project onto blocks of columns of $W$.

In [1]:
import numpy as np

from mpu_generators import MPUGenerator, UniformMPUGenerator
from TreeMPU import dense_unitary_from_tensors

tensors, l_vec, r_vec = UniformMPUGenerator(seed=7).branching(d=2, bond_dim=3, n_sites=4)
A = tensors[0]
print("site tensor shape:", A.shape, "| l =", l_vec.real, "| r =", r_vec.real)

site tensor shape: (2, 2, 3, 3) | l = [1. 0. 0.] | r = [1. 1. 1.]


## Checking the certificate by hand

The stored blocks are transposes of the operators: $A[v,u]$ as an operator is `A[:, :, v, u].T`.

In [2]:
d, _, chi, _ = A.shape
for v in range(chi):
    blocks = [A[:, :, v, u].T for u in range(chi) if np.any(A[:, :, v, u])]
    completeness = sum(B.conj().T @ B for B in blocks)
    cross = max(
        (
            np.linalg.norm(Bi.conj().T @ Bj)
            for i, Bi in enumerate(blocks)
            for j, Bj in enumerate(blocks)
            if i < j
        ),
        default=0.0,
    )
    print(
        f"row {v}: {len(blocks)} edges, ||sum B^dag B - 1|| = {np.linalg.norm(completeness - np.eye(d)):.1e}, "
        f"max ||B_i^dag B_j|| = {cross:.1e}"
    )

row 0: 2 edges, ||sum B^dag B - 1|| = 5.8e-16, max ||B_i^dag B_j|| = 2.0e-16
row 1: 2 edges, ||sum B^dag B - 1|| = 1.6e-15, max ||B_i^dag B_j|| = 1.7e-16
row 2: 2 edges, ||sum B^dag B - 1|| = 1.7e-15, max ||B_i^dag B_j|| = 3.4e-16


## Unitarity at every size, from one tensor

For `UniformMPUGenerator` the site tensor does not depend on `n_sites` (for a fixed seed), so the family $\{U_N\}$ is defined by one local datum. We verify unitarity densely for several $N$:

In [3]:
for N in (2, 3, 5, 8):
    t, l, r = UniformMPUGenerator(seed=7).branching(d=2, bond_dim=3, n_sites=N)
    assert np.array_equal(t[0], A)  # same tensor at every N
    U = dense_unitary_from_tensors(t, l, r)
    print(f"N={N}:  ||U^dag U - 1|| = {np.linalg.norm(U.conj().T @ U - np.eye(2**N)):.1e}")

N=2:  ||U^dag U - 1|| = 2.6e-15
N=3:  ||U^dag U - 1|| = 3.2e-15
N=5:  ||U^dag U - 1|| = 8.3e-15
N=8:  ||U^dag U - 1|| = 2.9e-14


## Site-dependent chains and combined readout

`MPUGenerator` draws a new digraph and Haar dressing at every site. `readout=[...]` reads out several bond values at once through extra cascade sites (thesis `cor:mode1-readout`, `rem:cascaded-readout`).

With `verify=True` (the default), every generated chain passes the algebraic checks. When $d^N\le2^{10}$ it also passes an exact dense unitarity check.

In [4]:
t, l, r = MPUGenerator(seed=3).branching(d=2, bond_dim=4, n_sites=4, readout=[0, 2, 3])
print("number of tensors (4 bulk + readout cascade):", len(t), "| boundary l =", l)
U = dense_unitary_from_tensors(t, l, r)
print("unitary:", np.allclose(U.conj().T @ U, np.eye(U.shape[0])))

number of tensors (4 bulk + readout cascade): 6 | boundary l = [1.+0.j]
unitary: True


## Blocking and full-rank boundaries

A chain whose boundary operators have full rank at every cut has minimal bond dimension. `TreeMPU.GeneralMPU(canonicalize=False)` requires this, and the canonical `UniformMPU` boundaries need the related Assumption 1. `blocking="auto"` returns the smallest block size, up to `max_blocking`, whose chain satisfies it.

In [5]:
from TreeMPU import satisfies_assumption_1

t, l, r = MPUGenerator(seed=0).branching(
    d=2, bond_dim=3, n_blocks=2, blocking="auto", max_blocking=3
)
print(
    "block size used:",
    int(np.log2(t[0].shape[0])),
    "| Assumption 1:",
    satisfies_assumption_1(t, l, r),
)

block size used: 2 | Assumption 1: True


**Next:** notebook 03 covers Mode II (fusion closure) and the named families.